# Tutorial for the YamboInput class

The `YamboInput` class creates, reads, modifies and writes the input files of Yambo (and of the other executables
of the package: ypp, yambo_rt, yambo_nl, ...). The class inherits from `dict`: the active runlevels are stored in the
`arguments` key and the variables in the `variables` one.

__The tutorial uses the nscf computation of the Tutorial_QeCalculator notebook, run it first.__

In [1]:
# useful to autoreload the module without restarting the kernel
%load_ext autoreload
%autoreload 2

import os, datetime
from mppi import InputFiles as I
from mppi.Calculators import Tools

print('Notebook executed on', datetime.datetime.now().strftime('%d/%m/%Y %H:%M'))

Notebook executed on 01/10/2026 12:03


## Preparation of the run_dir

Yambo needs the SAVE folder, built by `p2y` from the `.save` folder of a QuantumESPRESSO computation, and the
initialization (setup) of the run. The `init_yambo_dir` function creates the `yambo_dir` folder and runs `p2y` and the
setup in it (no operations are performed if the SAVE folder is already present)

In [2]:
run_dir = 'Yambo_tutorial' # shared with the Tutorial_YamboCalculator notebook
Tools.init_yambo_dir(yambo_dir=run_dir,input_dir='QeCalculator_tutorial/out_nscf/si_scf.save')

Create the folder path Yambo_tutorial
Executing command: cd Yambo_tutorial; p2y -I ../QeCalculator_tutorial/out_nscf/si_scf.save


Build the r_setup in the yambo_dir path Yambo_tutorial


## Generation of an input

The `args` parameter is the command line passed to yambo to generate the input file `filename` in the `folder`.
Here we build the input of a Hartree-Fock computation

In [3]:
inp = I.YamboInput(args='yambo -x -V rl',folder=run_dir,filename='hf.in')
inp

{'args': 'yambo -x -V rl',
 'folder': 'Yambo_tutorial',
 'filename': 'hf.in',
 'arguments': ['HF_and_locXC', 'infver'],
 'variables': {'FFTGvecs': [1139, 'RL'],
  'EXXRLvcs': [5985, 'RL'],
  'VXCRLvcs': [5985, 'RL'],
  'QPkrange': [[1, 32, 1, 8], '']}}

Variables with units, and arrays, are stored as a list `[value,units]`, while string variables are stored as
strings. The variables can be modified with the dictionary syntax or with the set methods of the class

In [4]:
inp.set_array_variables(units='Ha',EXXRLvcs=10.,VXCRLvcs=10.) # variables with the same units
inp.set_kRange(1,1)    # k points and bands of the QPkrange variable
inp.set_bandRange(4,5)
print(inp.convert_string())

HF_and_locXC
infver
FFTGvecs= 1139 RL
EXXRLvcs= 10.0 Ha
VXCRLvcs= 10.0 Ha
% QPkrange
1 | 1 | 4 | 5 |
%



The `write` method writes the input on file. If `args` is not empty and `reformat=True` (the default), yambo is run
again on the file to recover its original format (with the description of each variable). Note that in this case
yambo writes only the variables of the verbosity level given in `args` (for instance with `-V rl` the parallel
variables like `SE_CPU` are not written): use the proper verbosity (e.g. `-V par` or `-V all`) or `reformat=False`

In [5]:
inp.write(run_dir,'hf_modified.in')
print(open(os.path.join(run_dir,'hf_modified.in')).read().split('#\n',1)[1])

HF_and_locXC                     # [R] Hartree-Fock
infver                           # [R] Input file variables verbosity
FFTGvecs= 1139             RL    # [FFT] Plane-waves
EXXRLvcs= 10               Ha    # [XX] Exchange    RL components
VXCRLvcs= 10               Ha    # [XC] XCpotential RL components
%QPkrange                        # [GW] QP generalized Kpoint/Band indices
1|1|4|5|
%



## Reading an existing input

If `args` is empty the input file is read without running yambo (the SAVE folder is not needed, for instance to
manage the inputs of a computation performed on a remote machine)

In [6]:
inp2 = I.YamboInput(folder=run_dir,filename='hf_modified.in')
print(inp2['arguments'])
print(inp2['variables']['EXXRLvcs'],inp2['variables']['QPkrange'])

['HF_and_locXC', 'infver']
[10, 'Ha'] [[1, 1, 4, 5], '']


## Other runlevels

The same procedure applies to all the runlevels. For instance a GW computation in the plasmon pole approximation

In [7]:
inp = I.YamboInput(args='yambo -d p -p p -g n -V qp',folder=run_dir,filename='gw_ppa.in')
print(inp['arguments'])
inp.set_array_variables(BndsRnXp=[1,8],GbndRnge=[1,8])
inp.set_array_variables(units='Ry',NGsBlkXp=2.)
inp.set_bandRange(4,5)
{k:inp['variables'][k] for k in ['BndsRnXp','GbndRnge','NGsBlkXp','QPkrange']}

['ppa', 'gw0', 'el_el_corr', 'dyson', 'HF_and_locXC', 'em1d']


{'BndsRnXp': [[1, 8], ''],
 'GbndRnge': [[1, 8], ''],
 'NGsBlkXp': [2.0, 'Ry'],
 'QPkrange': [[1, 32, 4, 5], '']}

or the computation of the band structure with ypp. Arrays with more rows, like the k points of the path, are
lists of lists

In [8]:
inp = I.YamboInput(args='ypp -s b',folder=run_dir,filename='ypp_bands.in')
print(inp['arguments'])
inp.set_array_variables(BANDS_bands=[1,8])
inp.set_array_variables(BANDS_kpts=[[0.5,0.5,0.5],[0.,0.,0.],[0.,0.5,0.5]])
print(inp.convert_string())

['electrons', 'bnds']
electrons
bnds
PROJECT_mode= "none"
INTERP_mode= "NN"
INTERP_Shell_Fac= 20.0
INTERP_NofNN= 1
OutputAlat= 0.0
cooIn= "rlu"
cooOut= "rlu"
% BANDS_bands
1 | 8 |
%
CIRCUIT_E_DB_path= "none"
BANDS_path= ""
BANDS_steps= 10
% BANDS_kpts
0.5 | 0.5 | 0.5 |
0.0 | 0.0 | 0.0 |
0.0 | 0.5 | 0.5 |
%



The class also provides some methods to set the parameters of the real-time (`set_rt_field`, `set_rt_bands`,
`set_rt_simulationTimes`, ...) and ypp computations (`removeTimeReversal`, `set_ypp_extFields`).